# 第 27 课｜复权处理

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`27_youtube--H95z4T8cjI/index.html`  
> **所属模块**：P1-M6 · 数据契约与策略报告

## 本节量化问题

拆股、分红和配股怎样影响价格序列与历史收益？

## 准确定义与策略报告位置

原始价用于成交回放，复权价用于连续收益研究；调整因子必须匹配公司行动的已知时点。

- **核心关系**：总回报复权因子需同时反映拆股与现金分红；研究收益和成交回放必须保留不同价格列。
- **写入报告**：公司行动、原始价、复权价和可得时间
- **P2 衔接**：P2 第 35、39、43–44 课：时间、时点和接口口径。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 拆股日同时保留原始价与复权研究价

2 拆 1 会让原始价格机械减半，但持有者股数翻倍。复权价用于连续收益，原始价用于当时成交回放。

In [2]:
actions = pd.read_csv(DATA_DIR / "corporate_actions.csv", parse_dates=["date"])
raw = pd.Series(
    [100.0, 102.0, 51.5, 52.0],
    index=pd.to_datetime(["2025-01-02", "2025-01-03", "2025-01-06", "2025-01-07"]),
    name="raw_close",
)
split_date = actions.loc[actions["action"] == "split", "date"].iloc[0]
split_ratio = actions.loc[actions["action"] == "split", "value"].iloc[0]

adjusted = raw.copy()
adjusted.loc[adjusted.index < split_date] = adjusted.loc[adjusted.index < split_date] / split_ratio
comparison = pd.concat([raw, adjusted.rename("split_adjusted_close")], axis=1)
comparison["raw_return"] = comparison["raw_close"].pct_change(fill_method=None)
comparison["adjusted_return"] = comparison["split_adjusted_close"].pct_change(fill_method=None)
display(comparison)

,raw_close,split_adjusted_close,raw_return,adjusted_return
2025-01-02,100.0,50.0,NaN,NaN
2025-01-03,102.0,51.0,0.020000,0.020000
2025-01-06,51.5,51.5,-0.495098,0.009804
2025-01-07,52.0,52.0,0.009709,0.009709


## 方法边界与阶段验收

使用未来才知道的完整调整因子可能产生未来信息；信号和成交价不能混用口径。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
